# 03 — Nihai Model, Kilitli Test Tahminleri ve Bağımsız Test

**Bölüm A (kilitleme):** Nihai konfigürasyon tüm development üzerinde 3 seed ile eğitilir; test tahminleri **etiket sütunu olmadan**
`results/test_predictions_locked.csv` dosyasına yazılır. Bu bölümde test etiketleri okunmaz.

➜ Bölüm A bittikten sonra kilitli dosya **ayrı bir commit** ile GitHub'a yüklenir.

**Bölüm B (değerlendirme):** Yalnızca kilitli commit doğrulandıktan sonra test etiketleri okunur ve metrikler hesaplanır.

## 0. Kurulum

In [ ]:
# Google Colab'da çalıştırılıyorsa:
# !git clone https://github.com/mesutissever/ysa-midterm-mlp-nested-cv.git
# %cd ysa-midterm-mlp-nested-cv

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

from src.data import load_seeds, get_dev_test, get_test_features, TARGET
from src.final import (write_final_hyperparameters, train_final_models, write_locked_predictions,
                       check_lock_committed, LOCKED_PATH, THRESHOLD)

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
FIG, RES = "figures", "results"
def savefig(name): plt.savefig(f"{FIG}/{name}.png", dpi=150, bbox_inches="tight")
seeds = load_seeds()

# BÖLÜM A — Kilitleme (test etiketleri kullanılmaz)

## A1. Nihai konfigürasyon
Seçim, `02_nested_cv.ipynb` içinde tüm development iç CV'si ile önceden tanımlı kurala göre yapılmıştır.

In [ ]:
dev_sel = json.load(open(f"{RES}/dev_selection.json", encoding="utf-8"))
final_hp = write_final_hyperparameters(dev_sel, seeds)
print(json.dumps(final_hp, ensure_ascii=False, indent=1))
if final_hp["src_or_configs_uncommitted"]:
    print("\nUYARI: src/ veya configs/ altında commit edilmemiş değişiklik var. Kod sürümü kaydı için önce commit edin.")

## A2. Nihai eğitim: tüm development, 3 seed (ana seed: ilk seed) + LogisticRegression
Her seed için development içinden %15 stratified early-stop alt kümesi ayrılır (seed 31044); ön işleme yalnızca %85 fit alt kümesinde öğrenilir.

In [ ]:
df_dev, _ = get_dev_test()                       # test kısmı '_' — kullanılmıyor
config = final_hp["config"]
results, logreg, final_summary = train_final_models(df_dev, config, seeds)
final_summary

### Şekil 4b — Nihai modelin (ana seed) eğitim ve early-stop doğrulama kaybı; öğrenme oranı

In [ ]:
fh = pd.read_csv(f"{RES}/final_training_histories.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for s, g in fh.groupby("seed"):
    lw = 2.2 if s == final_hp["primary_seed"] else 1
    axes[0].plot(g.epoch, g.val_loss, lw=lw, label=f"early-stop doğrulama (seed {s})")
    axes[1].step(g.epoch, g.lr, where="post", lw=lw, label=f"seed {s}")
g = fh[fh.seed == final_hp["primary_seed"]]
axes[0].plot(g.epoch, g.train_loss, "k--", lw=1.2, label=f"eğitim (seed {final_hp['primary_seed']})")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE kaybı"); axes[0].legend(fontsize=8)
axes[1].set_yscale("log"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("öğrenme oranı"); axes[1].legend(fontsize=8)
fig.suptitle(f"Nihai model {final_hp['selected_code']} — tüm development eğitimi", y=1.02)
plt.tight_layout(); savefig("final_training_curves"); plt.show()

## A3. Kilitli test tahminleri (etiketsiz)
`get_test_features()` test kümesinin yalnızca özelliklerini döndürür; etiket sütunu bu aşamada okunmaz.

In [ ]:
X_test = get_test_features()
assert TARGET not in X_test.columns
locked, manifest = write_locked_predictions(results, logreg, X_test, seeds)
print(json.dumps(manifest, ensure_ascii=False, indent=1))
locked.head()

In [ ]:
# Tahmin dağılımı (etiketsiz betimsel bilgi)
locked.filter(like="prob_").describe().T.round(4)

---
## ⛔ DUR — Bölüm B'ye geçmeden önce commit et

```
git add results/test_predictions_locked.csv results/test_lock_manifest.json results/final_hyperparameters.json results/final_training_summary.csv results/final_training_histories.csv models/ figures/final_training_curves.png 03_final_test.ipynb
git commit -m "Kilitli test tahminleri (etiketsiz) - metrikler oncesi"
git push
```

Bölüm B'nin ilk hücresi bu commit'i doğrulamadan test etiketlerini okumaz.
Bu commit'ten sonra modele, eşiğe veya ön işlemeye müdahale edilmez.

# BÖLÜM B — Bağımsız test değerlendirmesi

*(Aşama 8'de eklenecek.)*